# Descarga del dataset NIH ChestX-ray14 a Google Drive

**Objetivo:** descargar a `Tesis_CXR/data/raw/nih/` los archivos necesarios para el Sprint 1:

1. `Data_Entry_2017.csv` (~8 MB) — labels de las 112,120 imágenes del dataset completo.
2. `images_001.tar.gz` (~1.9 GB) — primer tarball, contiene ~4999 imágenes.
3. Descomprimir el tar.gz → `data/raw/nih/images/*.png`

**Fuente:** Internet Archive (`archive.org/details/ChestXray-NIHCC`). Es un mirror oficialmente reconocido del NIH Box, con URLs estables.

**Tiempo estimado:** 15–30 min (depende del ancho de banda de Colab ese día).

**IMPORTANTE:** la descarga va directo a Drive. Si tu sesión de Colab se desconecta a la mitad, **el archivo parcial queda en Drive** — la celda de descarga detecta eso y reanuda/reintenta.


## 1. Montar Drive y definir rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
NIH_IMAGES_DIR = f'{NIH_DIR}/images'

Path(NIH_DIR).mkdir(parents=True, exist_ok=True)
Path(NIH_IMAGES_DIR).mkdir(parents=True, exist_ok=True)

print(f'NIH_DIR:        {NIH_DIR}')
print(f'NIH_IMAGES_DIR: {NIH_IMAGES_DIR}')
print(f'Existe NIH_DIR: {os.path.exists(NIH_DIR)}')


## 2. Verificar espacio disponible en Drive

Necesitamos al menos ~4 GB libres:
- 1.9 GB para `images_001.tar.gz`
- 1.9 GB para las imágenes descomprimidas (las `.png` ocupan similar)
- 8 MB para el CSV de labels

Cuando la descompresión termine, puedes borrar el `.tar.gz` y recuperar 1.9 GB.

In [ ]:
import shutil

total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Total: {total / (1024**3):.1f} GB')
print(f'Usado: {used  / (1024**3):.1f} GB')
print(f'Libre: {free  / (1024**3):.1f} GB')

if free < 4 * (1024**3):
    print('\n⚠ ADVERTENCIA: menos de 4 GB libres. Libera espacio antes de continuar.')
else:
    print('\n✓ Espacio suficiente para continuar.')


## 3. Descargar `Data_Entry_2017.csv`

Archivo chico (~8 MB), se descarga en segundos. Contiene los labels de las 112,120 imágenes del NIH completo (no solo las que vamos a usar).

In [ ]:
# FIX: URLs correctas de Internet Archive (tienen el path duplicado)
import urllib.request, os

BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'

# 1. CSV de labels
CSV_URL  = f'{BASE_URL}/Data_Entry_2017.csv'
CSV_PATH = f'{NIH_DIR}/Data_Entry_2017.csv'

if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 1_000_000:
    print(f'✓ Ya existe ({os.path.getsize(CSV_PATH)/(1024**2):.1f} MB)')
else:
    print(f'Descargando desde: {CSV_URL}')
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
    print(f'✓ Descargado ({os.path.getsize(CSV_PATH)/(1024**2):.1f} MB): {CSV_PATH}')

# 2. Actualizar la URL del tarball también (para la celda 5 posterior)
TAR_URL  = f'{BASE_URL}/images_001.tar.gz'
print(f'\n→ URL del tarball (para la siguiente celda): {TAR_URL}')

## 4. Validar CSV: ver estructura y patologías

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f'Filas: {len(df):,}')
print(f'Columnas: {df.columns.tolist()}')
print()
print('Primeras 3 filas:')
df.head(3)


In [ ]:
# Patologías únicas
all_labels = set()
for labels in df['Finding Labels']:
    for l in labels.split('|'):
        all_labels.add(l)

print(f'Patologías únicas ({len(all_labels)}):')
for l in sorted(all_labels):
    print(f'  - {l}')


In [ ]:
# Distribución general (dataset completo, no solo images_001)
print('DISTRIBUCIÓN EN EL DATASET COMPLETO (112,120 imágenes):')
print()
print(f'No Finding:   {(df["Finding Labels"] == "No Finding").sum():>6,}')
print(f'Cardiomegaly (sola o combinada): {df["Finding Labels"].str.contains("Cardiomegaly").sum():>6,}')
print(f'Cardiomegaly pura:               {(df["Finding Labels"] == "Cardiomegaly").sum():>6,}')
print()
print('⚠ Ojo: estos números son del dataset completo (112k imágenes).')
print('  En images_001 tendremos solo las primeras ~5000. La celda 8 nos dice')
print('  cuántos positivos quedan en nuestro subset.')


## 5. Descargar `images_001.tar.gz` (~1.9 GB)

Esta es la celda lenta. 15-30 min típicamente.

**Comportamiento:**
- Si ya existe el archivo y tiene el tamaño esperado → no re-descarga.
- Si existe pero es más chico (descarga parcial de sesión anterior) → reintenta desde cero.
- Usa `wget` con progreso visible en lugar de `urllib` porque es más robusto para archivos grandes.

In [ ]:
TAR_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC/images_001.tar.gz'
TAR_PATH = f'{NIH_DIR}/images_001.tar.gz'
EXPECTED_SIZE_BYTES = 1_900_000_000  # ~1.9 GB

need_download = True
if os.path.exists(TAR_PATH):
    size = os.path.getsize(TAR_PATH)
    size_gb = size / (1024**3)
    if size > EXPECTED_SIZE_BYTES * 0.95:  # al menos 95% del esperado
        print(f'✓ Ya existe y parece completo ({size_gb:.2f} GB): {TAR_PATH}')
        need_download = False
    else:
        print(f'⚠ Existe pero parece incompleto ({size_gb:.2f} GB). Re-descargando...')
        os.remove(TAR_PATH)

if need_download:
    print('Descargando images_001.tar.gz (~1.9 GB)...')
    print('Esto toma 15-30 min. No cierres la pestaña.')
    print()
    # -c: continua descargas interrumpidas. --progress=bar: barra visible.
    !wget -c --progress=bar:force:noscroll -O "{TAR_PATH}" "{TAR_URL}"
    size_gb = os.path.getsize(TAR_PATH) / (1024**3)
    print(f'\n✓ Descarga completa ({size_gb:.2f} GB)')


## 6. Verificar integridad del archivo

In [ ]:
import tarfile

try:
    with tarfile.open(TAR_PATH, 'r:gz') as tar:
        # Solo listar, no extraer todavía
        members = tar.getmembers()
    n_files = sum(1 for m in members if m.isfile())
    print(f'✓ Archivo íntegro. Contiene {n_files} archivos.')
    print(f'Primeros 3 miembros: {[m.name for m in members[:3]]}')
except Exception as e:
    print(f'✗ Archivo corrupto o incompleto: {e}')
    print('  Borra el archivo y vuelve a ejecutar la celda 5.')


## 7. Descomprimir las imágenes

**Tiempo:** 3-5 min.

El tar contiene una carpeta `images/` con ~4999 archivos `.png`. Extraemos directamente a `NIH_DIR` para que queden en `NIH_DIR/images/*.png`.

In [ ]:
import tarfile
from tqdm import tqdm

existing = os.listdir(NIH_IMAGES_DIR) if os.path.exists(NIH_IMAGES_DIR) else []
existing_pngs = [f for f in existing if f.endswith('.png')]

if len(existing_pngs) > 4000:
    print(f'✓ Ya hay {len(existing_pngs)} imágenes extraídas. Saltando.')
else:
    print('Descomprimiendo...')
    with tarfile.open(TAR_PATH, 'r:gz') as tar:
        members = [m for m in tar.getmembers() if m.isfile()]
        for m in tqdm(members, desc='Extrayendo'):
            tar.extract(m, path=NIH_DIR)
    print('✓ Descompresión completa.')

# Verificar
n_imgs = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png')])
print(f'\nTotal de imágenes en {NIH_IMAGES_DIR}: {n_imgs}')


## 8. Análisis del subset: ¿cuántos positivos nos quedan?

Este es el momento de la verdad: filtramos el CSV a solo las imágenes que descargamos y vemos cuántos casos de cada clase hay.

In [ ]:
# Lista de imágenes que tenemos físicamente en Drive
downloaded_imgs = set(f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png'))
print(f'Imágenes descargadas: {len(downloaded_imgs):,}')

# Filtrar el CSV a solo esas imágenes
df_subset = df[df['Image Index'].isin(downloaded_imgs)].copy()
print(f'Filas en CSV que matchean: {len(df_subset):,}')

if len(df_subset) == 0:
    print('✗ ERROR: ningún match. Revisa que las imágenes se hayan extraído bien.')
else:
    print()
    print('DISTRIBUCIÓN EN NUESTRO SUBSET:')
    print('-' * 60)
    n_no_finding  = (df_subset['Finding Labels'] == 'No Finding').sum()
    n_cardio_any  = df_subset['Finding Labels'].str.contains('Cardiomegaly').sum()
    n_cardio_pure = (df_subset['Finding Labels'] == 'Cardiomegaly').sum()

    print(f'No Finding:                       {n_no_finding:>5,}')
    print(f'Cardiomegaly (sola o combinada):  {n_cardio_any:>5,}')
    print(f'Cardiomegaly pura:                {n_cardio_pure:>5,}')
    print()

    if n_cardio_pure < 30:
        print('⚠ Pocos casos de Cardiomegaly pura en este subset.')
        print('  Recomendación: usar "Cardiomegaly sola o combinada" como positivo,')
        print('  o bajar también images_002.tar.gz para más volumen.')
    else:
        print(f'✓ Suficientes positivos para un baseline inicial.')


## 9. Guardar CSV del subset local

Para que el resto del pipeline no tenga que filtrar 112k filas cada vez, guardamos una versión reducida que solo tiene las imágenes que realmente tenemos.

In [ ]:
SUBSET_CSV_PATH = f'{NIH_DIR}/Data_Entry_subset_local.csv'
df_subset.to_csv(SUBSET_CSV_PATH, index=False)
print(f'✓ CSV del subset local guardado en: {SUBSET_CSV_PATH}')
print(f'  Filas: {len(df_subset):,}')


## 10. Resumen final

In [ ]:
print('='*60)
print('ESTADO FINAL DE LA DESCARGA NIH')
print('='*60)
print()
print(f'Carpeta NIH:        {NIH_DIR}')
print(f'  ├── Data_Entry_2017.csv         ({os.path.getsize(CSV_PATH) / (1024**2):.1f} MB)')
print(f'  ├── Data_Entry_subset_local.csv ({os.path.getsize(SUBSET_CSV_PATH) / (1024**2):.1f} MB)')
if os.path.exists(TAR_PATH):
    print(f'  ├── images_001.tar.gz           ({os.path.getsize(TAR_PATH) / (1024**3):.2f} GB) ← se puede borrar')
print(f'  └── images/')
n_imgs = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith(".png")])
print(f'       └── {n_imgs:,} imágenes .png')
print()
print('Siguiente paso: volver al chat para recibir:')
print('  1. scripts/prepare_splits.py (train/val/test estratificado por paciente)')
print('  2. src/datasets/transforms.py')
print('  3. src/datasets/nih.py')


## 11. (Opcional) Borrar el `.tar.gz` para liberar espacio

Si necesitas recuperar ~1.9 GB en Drive, puedes borrar el tarball. Las imágenes ya están extraídas.

In [ ]:
# Descomentar si quieres borrarlo
# os.remove(TAR_PATH)
# print(f'✓ {TAR_PATH} borrado')
